In [7]:
from pathlib import Path
import sys

# Point directly to the folder containing your dynacir directory
qiskit_workspace = Path(r"C:\Users\Raghav\OneDrive\Desktop\QML\Qiskit")

# Append it to sys.path if it isn't already there
if str(qiskit_workspace) not in sys.path:
    sys.path.insert(0, str(qiskit_workspace))

print(f"✅ Python path updated to scan: {qiskit_workspace}")

✅ Python path updated to scan: C:\Users\Raghav\OneDrive\Desktop\QML\Qiskit


In [9]:
from pathlib import Path

# Target the true path of your module
dynacir_path = Path(r"C:\Users\Raghav\OneDrive\Desktop\QML\Qiskit\dynacir")
passes_path = dynacir_path / "passes"

# Ensure the subfolder exists and create the required __init__.py files
passes_path.mkdir(parents=True, exist_ok=True)
(dynacir_path / "__init__.py").touch(exist_ok=True)
(passes_path / "__init__.py").touch(exist_ok=True)

print("📁 Package initialization files (__init__.py) verified and created!")

📁 Package initialization files (__init__.py) verified and created!


In [12]:
import os
from pathlib import Path

passes_dir = Path(r"C:\Users\Raghav\OneDrive\Desktop\QML\Qiskit\dynacir\passes")
print("Files in passes directory:", [f.name for f in passes_dir.glob("*.py")])

Files in passes directory: ['__init__.py']


In [13]:
from pathlib import Path

passes_dir = Path(r"C:\Users\Raghav\OneDrive\Desktop\QML\Qiskit\dynacir\passes")

# 1. Define the complete implementation for CollectResets
collect_resets_code = """from qiskit.transpiler.basepasses import AnalysisPass

class CollectResets(AnalysisPass):
    \"\"\"A custom analysis pass to collect and track mid-circuit reset/measurement operations.\"\"\"
    def __init__(self):
        super().__init__()
        self.resets = []

    def run(self, dag):
        self.resets.clear()
        for node in dag.op_nodes():
            if node.name == 'reset' or (node.name == 'measure' and getattr(node.op, 'condition', None) is not None):
                self.resets.append(node)
        self.property_set['collected_resets'] = self.resets
        return dag
"""

# 2. Write the file to disk
(passes_dir / "collect_resets.py").write_text(collect_resets_code)

# 3. Expose it via __init__.py for clean importing
(passes_dir / "__init__.py").write_text("from .collect_resets import CollectResets\n")

print("✨ CollectResets class successfully written and registered!")

✨ CollectResets class successfully written and registered!


In [23]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.circuit.controlflow import IfElseOp, WhileLoopOp

# ==========================================
# 1. Initialize and Configure Target Backend
# ==========================================
backend = GenericBackendV2(num_qubits=12)

# Explicitly register dynamic control flow operations to the target
backend.target.add_instruction(IfElseOp, name="if_else")
backend.target.add_instruction(WhileLoopOp, name="while_loop")

print(
    "✅ Target backend verified with operations:", list(backend.target.operation_names)
)


# ==========================================
# 2. Define Sub-Circuits for the Branches
# ==========================================
# For a structural if_else, we build explicit sub-circuits for the True and False paths.
# They must match the qubit and classical bit allocation of the targeting block.
true_body = QuantumCircuit(1)
true_body.x(0)  # Apply correction if condition met

false_body = QuantumCircuit(1)
false_body.id(0)  # Do nothing/identity if condition not met


# ==========================================
# 3. Generate TFIM Trotter Step Circuit
# ==========================================
def generate_tfim_trotter_step(
    num_qubits: int,
    J: float,
    g: float,
    dt: float,
    true_block: QuantumCircuit,
    false_block: QuantumCircuit,
) -> QuantumCircuit:
    """
    Generates a single Trotter step for a 1D TFIM chain with functional dynamic control flow.
    """
    qr = QuantumRegister(num_qubits, name="q")
    cr_measure = ClassicalRegister(num_qubits, name="mid_meas")
    qc = QuantumCircuit(qr, cr_measure)

    # --- Step A: Transverse Field Term (X-rotations) ---
    for i in range(num_qubits):
        qc.rx(2 * g * dt, qr[i])

    qc.barrier()

    # --- Step B: Ising Interaction Term (ZZ-rotations via interleaved pairs) ---
    for i in range(0, num_qubits - 1, 2):
        qc.cx(qr[i], qr[i + 1])
        qc.rz(2 * J * dt, qr[i + 1])
        qc.cx(qr[i], qr[i + 1])

    for i in range(1, num_qubits - 1, 2):
        qc.cx(qr[i], qr[i + 1])
        qc.rz(2 * J * dt, qr[i + 1])
        qc.cx(qr[i], qr[i + 1])

    qc.barrier()

    # --- Step C: Dynamic Mid-Circuit Measurement ---
    qc.measure(qr[0], cr_measure[0])

    # --- Step D: Functional Control Flow Application ---
    # Arguments expected by structural if_else:
    # (classical_cond, true_body, false_body, qubits, clbits)
    qc.if_else(
        (cr_measure[0], 1),
        true_block,
        false_block,
        [qr[0]],  # Qubits passed to the sub-blocks
        [],  # Classical bits passed to the sub-blocks (none needed here)
    )

    return qc


# ==========================================
# 4. Instantiate and Inspect Circuit
# ==========================================
num_qubits = 12
J_param = 1.0
g_param = 0.5
time_step = 0.05

tfim_circuit = generate_tfim_trotter_step(
    num_qubits, J_param, g_param, time_step, true_body, false_body
)

print(f"📦 TFIM circuit successfully built!")
print(f"Circuit Depth: {tfim_circuit.depth()}")
print(f"Circuit Operations Count: {dict(tfim_circuit.count_ops())}")

✅ Target backend verified with operations: ['cx', 'id', 'rz', 'sx', 'x', 'reset', 'delay', 'measure', 'if_else', 'while_loop']
📦 TFIM circuit successfully built!
Circuit Depth: 9
Circuit Operations Count: {'cx': 22, 'rx': 12, 'rz': 11, 'barrier': 2, 'measure': 1, 'if_else': 1}


In [24]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error


# ==========================================
# 1. Multi-Step Trotter Evolution Factory
# ==========================================
def generate_scaled_tfim_circuit(
    num_qubits: int, J: float, g: float, total_time: float, steps: int
) -> QuantumCircuit:
    """
    Composes a multi-step Trotter-Suzuki time evolution circuit for the TFIM
    with dynamic stabilization injected at each step interval.
    """
    dt = total_time / steps

    qr = QuantumRegister(num_qubits, name="q")
    cr_measure = ClassicalRegister(steps, name="mid_meas")  # Bit per step for tracking
    qc = QuantumCircuit(qr, cr_measure)

    # Pre-compiled sub-blocks for structural control flow
    true_body = QuantumCircuit(1)
    true_body.x(0)

    false_body = QuantumCircuit(1)
    false_body.id(0)

    for step in range(steps):
        # --- Transverse Field Term ---
        for i in range(num_qubits):
            qc.rx(2 * g * dt, qr[i])
        qc.barrier()

        # --- Ising Interaction Term ---
        for i in range(0, num_qubits - 1, 2):
            qc.cx(qr[i], qr[i + 1])
            qc.rz(2 * J * dt, qr[i + 1])
            qc.cx(qr[i], qr[i + 1])
        for i in range(1, num_qubits - 1, 2):
            qc.cx(qr[i], qr[i + 1])
            qc.rz(2 * J * dt, qr[i + 1])
            qc.cx(qr[i], qr[i + 1])
        qc.barrier()

        # --- Dynamic Stabilization Step ---
        # Measure boundary qubit into the specific classical bit for this step
        qc.measure(qr[0], cr_measure[step])
        qc.if_else((cr_measure[step], 1), true_body, false_body, [qr[0]], [])
        qc.barrier()

    return qc


# ==========================================
# 2. Build 10-Step Scaled Circuit
# ==========================================
total_evolution_time = 1.0
number_of_steps = 10

scaled_tfim_qc = generate_scaled_tfim_circuit(
    num_qubits=12, J=1.0, g=0.5, total_time=total_evolution_time, steps=number_of_steps
)
print(f"📈 Scaled TFIM Environment Built:")
print(f"Total Trotter Steps: {number_of_steps}")
print(f"Scaled Circuit Depth: {scaled_tfim_qc.depth()}")
print(f"Total CX Gates Placed: {scaled_tfim_qc.count_ops().get('cx', 0)}")

# ==========================================
# 3. Construct Local Aer Noise Model
# ==========================================
noise_model = NoiseModel()

# Parameters for typical superconducting qubit profiles
p_single = 0.001  # 0.1% single-qubit gate error rate
p_double = 0.01  # 1.0% two-qubit gate error rate

error_single = depolarizing_error(p_single, 1)
error_double = depolarizing_error(p_double, 2)

# Assign errors to the gate set natively
noise_model.add_all_qubit_quantum_error(error_single, ["rx", "rz", "x", "sx"])
noise_model.add_all_qubit_quantum_error(error_double, ["cx"])

# Instantiate an offline Aer Simulator configured with the custom target profile
sim_backend = AerSimulator(noise_model=noise_model)
print("\n🔒 Offline AerSimulator initialized with customized noisy gate targets.")

📈 Scaled TFIM Environment Built:
Total Trotter Steps: 10
Scaled Circuit Depth: 90
Total CX Gates Placed: 220

🔒 Offline AerSimulator initialized with customized noisy gate targets.


In [32]:
from qiskit.circuit.controlflow import ControlFlowOp
from qiskit.circuit import QuantumCircuit


def extract_dynamic_stabilizers(circuit: QuantumCircuit):
    """
    Directly extracts stabilizers from a QuantumCircuit layout,
    bypassing the PassManager and DAG tracking systems entirely.
    """
    collected_resets = []

    # Pre-map global qubits to clean index integers
    global_qubit_map = {qubit: idx for idx, qubit in enumerate(circuit.qubits)}

    def inspect_block(block, parent_global_indices):
        for inst, qargs, _ in block.data:
            # Map local block qubits to parent footprint positions positionally
            local_indices = [block.qubits.index(q) for q in qargs]
            global_indices = [parent_global_indices[idx] for idx in local_indices]

            if inst.name == "reset":
                collected_resets.append(("standard_reset", global_indices))
            elif inst.name == "x":
                collected_resets.append(("dynamic_stabilizer_x", global_indices))
            elif isinstance(inst, ControlFlowOp):
                for sub_block in inst.blocks:
                    inspect_block(sub_block, global_indices)

    # Walk the top-level circuit instruction stream
    for inst, qargs, _ in circuit.data:
        top_global_indices = [global_qubit_map[q] for q in qargs]

        if inst.name == "reset":
            collected_resets.append(("standard_reset", top_global_indices))
        elif inst.name == "x":
            collected_resets.append(("dynamic_stabilizer_x", top_global_indices))
        elif isinstance(inst, ControlFlowOp):
            for sub_block in inst.blocks:
                inspect_block(sub_block, top_global_indices)

    return collected_resets


# Run it directly on your 10-step TFIM circuit
collected = extract_dynamic_stabilizers(scaled_tfim_qc)

print(f"📊 --- Direct Extraction Results ---")
print(f"Total dynamic adjustments detected: {len(collected)}")
for idx, (op_type, target_qubits) in enumerate(collected):
    print(f"Step {idx+1}: Found {op_type} acting on global Qubit index {target_qubits}")

📊 --- Direct Extraction Results ---
Total dynamic adjustments detected: 10
Step 1: Found dynamic_stabilizer_x acting on global Qubit index [0]
Step 2: Found dynamic_stabilizer_x acting on global Qubit index [0]
Step 3: Found dynamic_stabilizer_x acting on global Qubit index [0]
Step 4: Found dynamic_stabilizer_x acting on global Qubit index [0]
Step 5: Found dynamic_stabilizer_x acting on global Qubit index [0]
Step 6: Found dynamic_stabilizer_x acting on global Qubit index [0]
Step 7: Found dynamic_stabilizer_x acting on global Qubit index [0]
Step 8: Found dynamic_stabilizer_x acting on global Qubit index [0]
Step 9: Found dynamic_stabilizer_x acting on global Qubit index [0]
Step 10: Found dynamic_stabilizer_x acting on global Qubit index [0]


C:\Users\Raghav\AppData\Local\Temp\ipykernel_15952\49523337.py:30: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for inst, qargs, _ in circuit.data:
C:\Users\Raghav\AppData\Local\Temp\ipykernel_15952\49523337.py:16: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for inst, qargs, _ in block.data:


In [33]:
from qiskit.circuit.controlflow import ControlFlowOp
from qiskit.circuit import QuantumCircuit


def extract_dynamic_stabilizers(circuit: QuantumCircuit):
    """
    Directly extracts stabilizers from a QuantumCircuit layout using modern
    Qiskit 1.x named attributes, clean of deprecation warnings.
    """
    collected_resets = []
    global_qubit_map = {qubit: idx for idx, qubit in enumerate(circuit.qubits)}

    def inspect_block(block, parent_global_indices):
        # Use named attributes (.operation, .qubits) instead of tuple unpacking
        for instruction in block.data:
            op = instruction.operation
            qargs = instruction.qubits

            local_indices = [block.qubits.index(q) for q in qargs]
            global_indices = [parent_global_indices[idx] for idx in local_indices]

            if op.name == "reset":
                collected_resets.append(("standard_reset", global_indices))
            elif op.name == "x":
                collected_resets.append(("dynamic_stabilizer_x", global_indices))
            elif isinstance(op, ControlFlowOp):
                for sub_block in op.blocks:
                    inspect_block(sub_block, global_indices)

    # Walk the top-level using the modern attributes API
    for instruction in circuit.data:
        op = instruction.operation
        qargs = instruction.qubits

        top_global_indices = [global_qubit_map[q] for q in qargs]

        if op.name == "reset":
            collected_resets.append(("standard_reset", top_global_indices))
        elif op.name == "x":
            collected_resets.append(("dynamic_stabilizer_x", top_global_indices))
        elif isinstance(op, ControlFlowOp):
            for sub_block in op.blocks:
                inspect_block(sub_block, top_global_indices)

    return collected_resets


# Execute the clean function
collected = extract_dynamic_stabilizers(scaled_tfim_qc)
print(f"📊 --- Cleaned Direct Extraction Results ---")
print(f"Total dynamic adjustments detected: {len(collected)}")

📊 --- Cleaned Direct Extraction Results ---
Total dynamic adjustments detected: 10


In [34]:
from qiskit import transpile

# ==========================================
# 1. Transpile for the Noisy Simulator Target
# ==========================================
# We must transpile the circuit to match the simulator's basic gate capabilities
# while explicitly informing it to preserve the dynamic control flow operations.
transpiled_tfim_qc = transpile(
    scaled_tfim_qc, backend=sim_backend, optimization_level=1
)

print("⚛️ Transpilation complete.")
print(f"Final Operations Profile: {dict(transpiled_tfim_qc.count_ops())}")

# ==========================================
# 2. Execute the Noisy Simulation Run
# ==========================================
# Run the simulation utilizing our pre-configured noisy backend target
shots_count = 2048
print(f"⏳ Executing noisy simulation run ({shots_count} shots)...")

sim_job = sim_backend.run(transpiled_tfim_qc, shots=shots_count)
sim_result = sim_job.result()

# Extract the final measurement statistics
output_counts = sim_result.get_counts()

print("\n📊 --- Noisy Simulation Execution Results ---")
print(f"Unique output state configurations tracked: {len(output_counts)}")
# Display the top 5 highest frequency measurement results to check stability
sorted_counts = sorted(output_counts.items(), key=lambda item: item[1], reverse=True)
print("Top 5 State Configurations (Bitstring : Occurrences):")
for bitstring, count in sorted_counts[:5]:
    print(f"  {bitstring} : {count}")

⚛️ Transpilation complete.
Final Operations Profile: {'cx': 220, 'rx': 120, 'rz': 110, 'barrier': 30, 'measure': 10, 'if_else': 10}
⏳ Executing noisy simulation run (2048 shots)...

📊 --- Noisy Simulation Execution Results ---
Unique output state configurations tracked: 23
Top 5 State Configurations (Bitstring : Occurrences):
  0000000000 : 1785
  0100000000 : 34
  0000000100 : 30
  0000100000 : 29
  0000000001 : 28
